<a href="https://colab.research.google.com/github/adityahadagalle/ML_practise/blob/main/Feature_Engg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telco Customer Churn Prediction

## Setup and Library Installation

In [30]:
pip install imbalanced-learn

## Data Loading and Initial Cleaning

In [31]:
import pandas as pd

# Load the specified CSV file
df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn (1).csv')

# Drop 'customerID' as it is a unique identifier and not useful for modeling
df=df.drop("customerID",axis=1)

# Convert 'Churn' column to numeric (1 for 'Yes', 0 for 'No')
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Convert 'TotalCharges' to numeric, coercing errors to NaN
# This handles cases where 'TotalCharges' might contain non-numeric strings (e.g., empty strings)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Display the first 5 rows of the DataFrame to verify loading and initial cleaning
display(df.head())

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## Feature Engineering

In [32]:
import pandas as pd

# Binary features based on tenure, contract, and service status
df['is_new_customer'] = (df['tenure'] < 6).astype(int)
df['is_loyal_customer'] = (df['tenure'] > 24).astype(int)
df['is_month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)
df['no_security'] = (df['OnlineSecurity'] == 'No').astype(int)
df['no_support'] = (df['TechSupport'] == 'No').astype(int)

# Calculate total number of services subscribed by a customer
services = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
            "TechSupport", "StreamingTV", "StreamingMovies"]
df["Total_Services"] = (df[services] == "Yes").sum(axis=1)
df['low_engagement'] = (df['Total_Services'] <= 2).astype(int)

# Interaction features combining service types and support
df["fiber_no_support"] = (
    (df["InternetService"] == "Fiber optic") &
    (df["TechSupport"] == "No")
)
df["new_no_support"] = (df["tenure"] < 6) & (df["TechSupport"] == "No")
df["senior_no_support"] = (
    (df["SeniorCitizen"] == 1) &
    (df["TechSupport"] == "No")
)
df["female_no_support"] = (
    (df["gender"] == "Female") &
    (df["TechSupport"] == "No")
)

# Financial features
# Note: 'TotalCharges' can have NaN values after coercion, handle this before division
# Replacing 0 with pd.NA to ensure division by zero results in NaN, not inf
df["charge_ratio"] = df["MonthlyCharges"] / df["TotalCharges"].replace(0, pd.NA)
df['Expected_Total'] = df['MonthlyCharges'] * df['tenure']

# Complex interaction feature
df["new_no_support_high_charge"] = (
    (df["tenure"] < 6) &
    (df["TechSupport"] == "No") &
    (df["MonthlyCharges"] > df["MonthlyCharges"].median())
)

# Cost per service (add 1 to Total_Services to avoid division by zero)
df['Cost_Per_Service'] = df['MonthlyCharges'] / (df['Total_Services'] + 1)

## Feature Validation: Churn Rates for Engineered Features

In [33]:
print("Churn Rate for Fiber Optic customers without Tech Support:")
print(df.groupby("fiber_no_support")["Churn"].mean())

print("\nCount of Fiber Optic customers without Tech Support:")
print(df["fiber_no_support"].value_counts())

Churn Rate for Fiber Optic customers without Tech Support:
fiber_no_support
False    0.159568
True     0.493722
Name: Churn, dtype: float64

Count of Fiber Optic customers without Tech Support:
fiber_no_support
False    4813
True     2230
Name: count, dtype: int64


In [34]:
print("Churn Rate for New Customers (tenure < 6) without Tech Support:")
print(df.groupby("new_no_support")["Churn"].mean())

Churn Rate for New Customers (tenure < 6) without Tech Support:
new_no_support
False    0.206194
True     0.665198
Name: Churn, dtype: float64


In [35]:
# Validate churn rate for 'senior_no_support' and 'female_no_support'
for col in ["senior_no_support", "female_no_support"]:
    print(f"\nFeature: {col}")
    print("Churn Rate:")
    print(df.groupby(col)["Churn"].mean())
    print("\nCount:")
    print(df[col].value_counts())


Feature: senior_no_support
Churn Rate:
senior_no_support
False    0.233221
True     0.506024
Name: Churn, dtype: float64

Count:
senior_no_support
False    6213
True      830
Name: count, dtype: int64

Feature: female_no_support
Churn Rate:
female_no_support
False    0.215988
True     0.418903
Name: Churn, dtype: float64

Count:
female_no_support
False    5329
True     1714
Name: count, dtype: int64


## Exploratory Data Analysis & Correlations

In [36]:
print("Correlation of Features with Churn (Numeric Only):")
print(df.corr(numeric_only=True)["Churn"].sort_values())

Correlation of Features with Churn (Numeric Only):
tenure                       -0.352229
is_loyal_customer            -0.309386
TotalCharges                 -0.199484
Expected_Total               -0.198514
Total_Services               -0.087698
low_engagement                0.110010
SeniorCitizen                 0.150889
MonthlyCharges                0.193356
female_no_support             0.197208
senior_no_support             0.199215
new_no_support_high_charge    0.286608
is_new_customer               0.308773
charge_ratio                  0.318539
Cost_Per_Service              0.319316
no_support                    0.337281
no_security                   0.342637
new_no_support                0.348377
fiber_no_support              0.352038
is_month_to_month             0.405103
Churn                         1.000000
Name: Churn, dtype: float64


In [37]:
print("Churn Rate by Contract Type:")
print(df.groupby("Contract")["Churn"].mean())

Churn Rate by Contract Type:
Contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: Churn, dtype: float64


In [38]:
print("Average Monthly Charges by Churn Status:")
print(df.groupby("Churn")["MonthlyCharges"].mean())

Average Monthly Charges by Churn Status:
Churn
0    61.265124
1    74.441332
Name: MonthlyCharges, dtype: float64


VALIDATION

In [41]:
df["fiber_no_support"] = (
    (df["InternetService"] == "Fiber optic") &
    (df["TechSupport"] == "No")
)
print("Churn Rate:")
print(df.groupby("fiber_no_support")["Churn"].mean())

print("\nCount:")
print(df["fiber_no_support"].value_counts())

Churn Rate:
fiber_no_support
False    0.159568
True     0.493722
Name: Churn, dtype: float64

Count:
fiber_no_support
False    4813
True     2230
Name: count, dtype: int64


In [25]:
df["new_no_support"] = (df["tenure"] < 6) & (df["TechSupport"] == "No")
df.groupby("new_no_support")["Churn"].mean()

,Churn
new_no_support,
False,0.206194
True,0.665198


In [32]:
df["senior_no_support"] = (
    (df["SeniorCitizen"] == 1) &
    (df["TechSupport"] == "No")
)

df["female_no_support"] = (
    (df["gender"] == "Female") &
    (df["TechSupport"] == "No")
)



In [34]:
# Validate churn rate for each feature
for col in ["senior_no_support", "female_no_support"]:
    print(f"\nFeature: {col}")
    print("Churn Rate:")
    print(df.groupby(col)["Churn"].mean())
    print("\nCount:")
    print(df[col].value_counts())


Feature: senior_no_support
Churn Rate:
senior_no_support
False    0.233221
True     0.506024
Name: Churn, dtype: float64

Count:
senior_no_support
False    6213
True      830
Name: count, dtype: int64

Feature: female_no_support
Churn Rate:
female_no_support
False    0.215988
True     0.418903
Name: Churn, dtype: float64

Count:
female_no_support
False    5329
True     1714
Name: count, dtype: int64


In [35]:
df.corr(numeric_only=True)["Churn"].sort_values()

,Churn
tenure,-0.352229
TotalCharges,-0.199484
Expected_Total,-0.198514
Total_Services,-0.087698
high_total_no_support,0.020882
senior_female_no_support,0.134191
SeniorCitizen,0.150889
MonthlyCharges,0.193356
female_no_support,0.197208
senior_no_support,0.199215


In [14]:
df.groupby("Contract")["Churn"].mean()

,Churn
Contract,
Month-to-month,0.427097
One year,0.112695
Two year,0.028319


In [15]:
df.groupby("Churn")["MonthlyCharges"].mean()

,MonthlyCharges
Churn,
0,61.265124
1,74.441332
